In [4]:
# We have added LangSmith's API key, endpoint and project's name in .env.chatbot file 
# So just load that particular files using load_dotenv()
# We don't need to make any changes in backend file.

import os
from dotenv import load_dotenv

from langchain_core.messages import HumanMessage

from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI

from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages

from langgraph.prebuilt import ToolNode, tools_condition


import asyncio
from tools import google_search, weather_search, fetch_train_details

print("RUNNING UPDATED FILE ✅")


load_dotenv(".env.chatbot")    # Serper_API_KEY is added in .env.chatbot file
os.environ['LANGSMITH_PROJECT'] = 'Chatbot_2'

# Creating the model
model_openai = ChatOpenAI(
    model = 'gpt-3.5-turbo'
)

#----------------------------------------------------------------------------------------------------------------------
# Creating Tools


#----------------------------------------------------------------------------------------------------------------------
# Binding tools
tools_list = [google_search, weather_search, fetch_train_details]
model_with_tools = model_openai.bind_tools(tools_list)
#----------------------------------------------------------------------------------------------------------------------

# Defining the state
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

#----------------------------------------------------------------------------------------------------------------------

def build_graph():

    # Defining nodes
    def chat_node(state: ChatState) ->ChatState:

        # Fetching messages
        messages = state["messages"]

        # Invokig model
        response = model_with_tools.invoke(messages)

        # Appending AI's response to the messages list
        return {"messages": [response]}

    # tool node
    tools_node = ToolNode(tools_list)

    # Defining graph
    graph = StateGraph(ChatState)

    # Adding nodes
    graph.add_node('chat_node', chat_node)
    graph.add_node("tools", tools_node)

    # Creating edges
    graph.add_edge(START, 'chat_node')
    graph.add_conditional_edges('chat_node', tools_condition)
    graph.add_edge('tools', 'chat_node')

    # Compiling the graph
    workflow = graph.compile(name = 'Streamlit chatbot')

    return workflow

#----------------------------------------------------------------------------------------------------------------------

def main():

    chatbot = build_graph()

    question = {'messages': [HumanMessage(content = 'When train_no 22959 reaches Rajkot.')]}
    response = chatbot.invoke(question)

    return response["messages"][-1].content


print(main())

RUNNING UPDATED FILE ✅
Train number 22959 (INTERCITY EXP) reaches Rajkot Junction (RJT) at 21:44 and departs at 21:49.


Nikunj
